# Design an Online Collaborative Spreadsheet

**Company:** MongoDB (GothamLoop question bank) · **Category:** System Design · **Tags:** Onsite Loop, API Design, Caching, Concurrency, Databases, Distributed Systems · **Difficulty/Frequency:** Uncommon (3/10)

> **Related:** [`1. Chat_Application_WhatsApp`](../1.%20Chat_Application_WhatsApp/1.%20Chat_Application_WhatsApp.ipynb) is fan-out done properly; [`11. Task_Scheduler`](../../2.%20Coding_Questions/11.%20Task_Scheduler/11.%20Task_Scheduler.ipynb) is the topological sort behind formula dependencies.

## Concepts

**What this question is really testing:** whether you build an **operation-based** system rather than a state-based one — and whether you notice what a *total order* does and does not buy you.

**First-principles primer:**

- **Operations, not state.** Don't send "here is the document"; send "B2 became `hello`". The document is then *derived* by replaying operations. That one decision hands you durability (persist the log), ordering, conflict resolution, and time travel — all from the same structure.
- **The cell is the unit of change.** Not the document, not the row. Two people editing different cells then have no conflict at all, and messages stay ~400 bytes instead of megabytes.
- **A server-assigned sequence number** gives every operation a place in a single total order, per document. That is what makes "who wins?" answerable without consensus: **higher `seq` wins**. One atomic counter replaces an entire distributed-agreement protocol.
- **Snapshot + log** is how the log stops growing forever. Periodically write the full cell state; on load, fetch the snapshot plus operations after it.

**Simple worked example.** Alice and Bob both edit B2 at "the same time":

```
Alice's browser:  setCell(B2, "cat")  --sent-->
                                                 server assigns seq 41  --> everyone
Bob's browser:    setCell(B2, "dog")  --sent-->
                                                 server assigns seq 42  --> everyone

Both clients apply 41 then 42.  Final state: B2 = "dog".
Alice's screen showed "cat" for ~40ms (optimistic apply), then flipped to "dog".
```

Nobody negotiated. The counter decided.

**The question to keep asking:** `seq` records **when the server heard about an operation**, not when a human made it. While everyone is online those are nearly the same thing. The moment someone goes offline, they diverge — and every property built on `seq` inherits that gap. Hold that thought.

## Requirements & Scale

| Functional | Non-functional |
|---|---|
| Create / view / edit spreadsheets | Real-time propagation |
| Simultaneous multi-user editing | Handles concurrent same-cell edits |
| Data, formulas, formatting | Offline edits + reconnection |
| | Scale to many users and documents |

**Stated scale:** 10M DAU × 50 edits/day = 500M ops/day; 100 concurrent editors on the hottest document.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "capacity.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from capacity import (DAY, KB, MB, GB, TB, human_bytes, human_count,
                      human_rate, table, assumption_table)

DAU = 10_000_000
EDITS_PER_USER = 50
PEAK_LOW, PEAK_HIGH = 3, 5
OP_BYTES = 400
MAX_CONCURRENT_EDITORS = 100

assumption_table({
    "Daily active users":  human_count(DAU),
    "Edits per user/day":  EDITS_PER_USER,
    "Peak multiplier":     f"{PEAK_LOW}-{PEAK_HIGH}x",
    "Bytes per operation": f"{OP_BYTES} B  (stated 200-500)",
    "Hottest document":    f"{MAX_CONCURRENT_EDITORS} concurrent editors",
})


def bps(n):
    return human_bytes(n) + "/s"


ops_per_day = DAU * EDITS_PER_USER
ops_avg = ops_per_day / DAY
ops_peak = ops_avg * PEAK_HIGH
log_per_day = ops_per_day * OP_BYTES

table([
    ("Operations / day",     human_count(ops_per_day)),
    ("Ops/sec average",      human_rate(ops_avg, " ops/s")),
    (f"Ops/sec peak ({PEAK_LOW}-{PEAK_HIGH}x)",
     f"{ops_avg * PEAK_LOW / 1000:.1f}k - {ops_peak / 1000:.1f}k ops/s"),
    ("", ""),
    ("Log growth / day",     human_bytes(log_per_day)),
    ("Log growth / year",    human_bytes(log_per_day * 365)),
], title="OPERATION VOLUME")

assert ops_per_day == 500_000_000
assert 5_700 < ops_avg < 5_900, "the answer's stated ~5,800 ops/s"
assert abs(log_per_day - 200 * GB) / GB < 1, "the answer's stated 200 GB/day"
assert abs(log_per_day * 365 - 73 * TB) / TB < 1, "the answer's stated ~73 TB/year"

# A small honesty note on the peak figure.
print(f"\n  Note: 3x average is {ops_avg * PEAK_LOW:,.0f} ops/s, not the 'about 20,000'")
print(f"  the answer rounds to. The 5x figure ({ops_peak:,.0f}) is the one it then uses,")
print("  so the bandwidth line below is computed at 30k. Minor, but derive the")
print("  range from the multiplier rather than reaching for round numbers.")

### ⚠️ Problem 3 — the bandwidth estimate ignores fan-out

> *"At 30,000 ops/sec × 400 bytes ≈ 12 MB/s of operation traffic. That's trivially handled by a handful of relay servers."*

12 MB/s is **ingress**. Every operation is then *broadcast to every other subscriber of that document* — which is the entire point of a collaborative app. Egress is `(C − 1) ×` ingress for `C` concurrent collaborators.

The answer names the 100-editor document in the very next bullet and never multiplies it through.

In [ ]:
INGRESS_OPS = 30_000                       # the answer's own working figure
ingress = INGRESS_OPS * OP_BYTES
assert abs(ingress - 12 * MB) / MB < 0.1, "the answer's stated ~12 MB/s"

rows = [("INGRESS (as stated)", bps(ingress))]
rows.append(("", ""))
for c in (2, 5, 10, 100):
    rows.append((f"EGRESS at {c:>3} collaborators", f"{bps(ingress * (c - 1)):>10}"
                                                    f"   ({c - 1}x)"))
table(rows, title="FAN-OUT: WHAT THE RELAYS ACTUALLY SEND")

assert ingress * (MAX_CONCURRENT_EDITORS - 1) > 1 * GB, \
    "the stated 100-editor case is over 1 GB/s of egress"

NIC_10GBPS = 1.25 * GB
print(f"\n  => At 2 collaborators the answer's number is right. At the 100 the answer")
print(f"     itself names, egress is {bps(ingress * 99)} - {ingress * 99 / NIC_10GBPS:.0%} of a 10 Gb/s NIC,")
print("     for ONE document.")
print("\n     This is fan-out on write, exactly as in the chat problem - where it is")
print("     the headline insight. Here it is dropped. The design still works")
print("     (relays are stateless and scale horizontally), but the fan-out factor")
print("     is what sizes the fleet, so it is the wrong number to say out loud.")

# Fan-out also decides the compaction story.
print()
table([
    ("Log growth / day",        human_bytes(log_per_day)),
    ("...but egress / day",     human_bytes(log_per_day * 4)),
    ("", ""),
    ("Log is written",          "ONCE per operation"),
    ("Log is sent",             "(C-1) times per operation"),
], title="WHY STORAGE AND BANDWIDTH SCALE DIFFERENTLY")
print("\n  => Storage grows with ops; bandwidth grows with ops x collaborators.")
print("     Compaction fixes the first and does nothing for the second.")

## The operation log and the materialized view

The log is the source of truth; `cells` is a materialized view. Building it, and then finding out what the schema is missing.

In [ ]:
from typing import Dict, Tuple, List, Optional
from dataclasses import dataclass, field

Cell = Tuple[str, int, int]          # (sheet_id, row, col)


@dataclass
class Op:
    seq: Optional[int]
    user: str
    cell: Cell
    value: str
    base_seq: int = 0                # the seq this edit was made against
    op_id: str = ""


class Doc:
    """The materialized view. `track_seq` toggles the missing schema column."""

    def __init__(self, track_seq: bool):
        self.values: Dict[Cell, str] = {}
        self.cell_seq: Dict[Cell, int] = {}
        self.track_seq = track_seq

    def apply(self, op: Op) -> bool:
        if self.track_seq:
            if op.seq <= self.cell_seq.get(op.cell, -1):
                return False                       # stale or duplicate: ignore
            self.cell_seq[op.cell] = op.seq
        self.values[op.cell] = op.value            # blind overwrite without seq
        return True


class Server:
    def __init__(self):
        self.log: List[Op] = []
        self.next_seq = 1

    def submit(self, op: Op) -> Op:
        op.seq = self.next_seq
        self.next_seq += 1
        self.log.append(op)
        return op


B2: Cell = ("Sheet1", 2, 2)

srv = Server()
doc = Doc(track_seq=True)
for user, val in [("alice", "cat"), ("bob", "dog")]:
    doc.apply(srv.submit(Op(None, user, B2, val)))

table([("Log length", len(srv.log)),
       ("B2 value",   doc.values[B2]),
       ("B2 seq",     doc.cell_seq[B2])],
      title="TWO CONCURRENT EDITS TO B2")
assert doc.values[B2] == "dog", "higher seq wins - the counter decided, not a lock"
print("\n  No lock, no consensus. One atomic counter resolved it.")

### ⚠️ Problem 2 — `cells` has no `seq` column, so LWW cannot be enforced there

The conflict rule is *"the operation with the higher `seq` wins"*. But the `cells` table stores only:

```sql
value, format, updated_at, updated_by      -- no seq
```

The materialized view has no record of **which operation produced its current value**. So a materializer, a recovering replica, or a client filling a gap cannot tell whether an incoming operation is newer or older than what it already holds. Any out-of-order or duplicate delivery corrupts the state.

This also disposes of the claim that *"operations are idempotent and commutative enough for our purposes."* Without the column they are **neither**. With it, they genuinely become both.

In [ ]:
import random

ops = [Op(None, "u", B2, v) for v in ("first", "second", "third", "fourth")]
srv = Server()
ordered = [srv.submit(o) for o in ops]
TRUTH = "fourth"                                   # highest seq

# --- Delivered in order: both versions look fine. ---
for track in (False, True):
    d = Doc(track_seq=track)
    for o in ordered:
        d.apply(o)
    assert d.values[B2] == TRUTH

# --- Delivered out of order (a gap-fill, a replica catching up, a retry). ---
rows = []
for track in (False, True):
    wrong = 0
    for trial in range(200):
        shuffled = ordered[:]
        random.Random(trial).shuffle(shuffled)
        d = Doc(track_seq=track)
        for o in shuffled:
            d.apply(o)
        wrong += (d.values[B2] != TRUTH)
    rows.append((f"seq column {'PRESENT' if track else 'ABSENT '}",
                 f"{wrong:3d} / 200 orderings give the WRONG value"))
table(rows, title="OUT-OF-ORDER DELIVERY OF THE SAME 4 OPERATIONS")

assert rows[0][1].startswith("1"), "without seq, most orderings corrupt the cell"
assert rows[1][1].startswith("  0"), "with seq, every ordering converges"

# Commutativity and idempotence, stated as properties rather than asserted in prose.
d = Doc(track_seq=True)
for o in ordered:
    d.apply(o)
for o in ordered * 3:                              # replay everything three more times
    d.apply(o)
assert d.values[B2] == TRUTH, "idempotent: replay changes nothing"

print("\n  With a per-cell seq and a max-wins apply, the operations are genuinely")
print("  commutative AND idempotent - which is what makes replay, recovery and")
print("  at-least-once delivery safe. Without it, 'commutative enough' is just wrong.")
print("\n  The fix is one column:")
print("    ALTER TABLE cells ADD COLUMN seq BIGINT NOT NULL;")
print("    -- apply: if op.seq > cell.seq: cell.value, cell.seq = op.value, op.seq")

### ⚠️ Problem 1 — offline reconnection silently destroys newer work

> *"On reconnect, the client sends all pending operations to the server. **The server assigns `seq` numbers in the order received**."*

`seq` records when the server *heard* about an operation. For an offline client that is 30 minutes after the edit was made — so the stale edit receives the **highest** `seq` of all, and LWW hands it the win.

**The person who was offline longest wins.**

In [ ]:
def scenario(policy: str):
    """Alice edits offline at 09:00; Bob and Carol edit later; Alice reconnects."""
    srv = Server()
    doc = Doc(track_seq=True)

    alice_pending = Op(None, "alice", B2, "old draft", base_seq=0, op_id="a1")

    doc.apply(srv.submit(Op(None, "bob", B2, "reviewed", base_seq=0)))
    doc.apply(srv.submit(Op(None, "carol", B2, "final, approved by legal",
                            base_seq=1)))
    before_reconnect = doc.values[B2]

    conflicts = []
    if policy == "as_written":
        doc.apply(srv.submit(alice_pending))               # seq assigned on ARRIVAL
    elif policy == "base_seq_check":
        current = doc.cell_seq.get(B2, -1)
        if alice_pending.base_seq < current:
            conflicts.append((alice_pending.value, doc.values[B2]))  # surfaced, not applied
        else:
            doc.apply(srv.submit(alice_pending))
    return before_reconnect, doc.values[B2], conflicts


before, after, _ = scenario("as_written")
table([("09:05 Bob",          "reviewed"),
       ("09:20 Carol",        "final, approved by legal"),
       ("State before Alice", before),
       ("", ""),
       ("09:30 Alice reconnects", "her 09:00 edit gets the HIGHEST seq"),
       ("FINAL VALUE",        after)],
      title="AS WRITTEN: seq ASSIGNED ON ARRIVAL")

assert before == "final, approved by legal"
assert after == "old draft", "a 30-minute-old edit overwrote two newer ones"
print("\n  => 30 minutes of reviewed work, silently replaced. No warning to anyone.")
print("     Calling this a 'rebase' obscures it: git's rebase STOPS on a conflict.")
print("     This one applies.")

before, after, conflicts = scenario("base_seq_check")
table([("State before Alice", before),
       ("FINAL VALUE",        after),
       ("Conflicts surfaced", str(conflicts))],
      title="WITH baseSeq: THE STALE EDIT IS CAUGHT")

assert after == "final, approved by legal", "newer work survives"
assert len(conflicts) == 1, "and Alice is told, rather than silently overruled"
print("\n  Each pending op carries the seq it was made against:")
print('    {"type": "setCell", "cell": {...}, "value": {...}, "baseSeq": 99}')
print("  If the cell has moved on, the server must not apply it silently.")

In [ ]:
# Which policy you pick is a product decision - name it explicitly.
table([
    ("Reject + surface",   "show both values, let the user choose (Google Sheets)"),
    ("Keep both",          "write the loser into a comment / 'conflicting copy'"),
    ("LWW by wall clock",  "compare timestamp, not seq - but client clocks lie"),
    ("Blind LWW by seq",   "as written: silent data loss"),
], title="WHAT TO DO WITH A STALE OFFLINE EDIT")

# Note that the FIRST three all preserve the data; only the last discards it.
print("\n  => LWW is not the mistake. LWW keyed on ARRIVAL order is a different rule")
print("     from LWW keyed on EDIT order, and only one matches what a user expects.")
print("\n  Cells the offline client did NOT touch are unaffected either way - which is")
print("  why this is easy to miss in testing: it only bites on the same cell.")

# Offline edits to untouched cells rebase cleanly, as the answer claims.
srv, doc = Server(), Doc(track_seq=True)
A1, C3 = ("Sheet1", 1, 1), ("Sheet1", 3, 3)
doc.apply(srv.submit(Op(None, "bob", A1, "bob's work", base_seq=0)))
pending = Op(None, "alice", C3, "alice's offline work", base_seq=0)
cur = doc.cell_seq.get(C3, -1)
assert pending.base_seq >= cur, "different cell: no conflict, applies cleanly"
doc.apply(srv.submit(pending))
assert doc.values[A1] == "bob's work" and doc.values[C3] == "alice's offline work"
print("\n  Different cells: both survive. The 'rebase is usually trivial' claim holds")
print("  for everything EXCEPT the case the conflict rule exists to handle.")

## Snapshots: why the log has to be compacted

500M ops/day is 200 GB/day and 73 TB/year. Loading a document by replaying its whole history is not viable, and neither is keeping the log forever.

The state, by contrast, is tiny — a sparse map of non-empty cells. That gap is the whole argument for snapshotting.

In [ ]:
CELLS_PER_DOC = 5_000                 # non-empty cells in a typical sheet
CELL_STATE_BYTES = 150
DOCS = 20_000_000
EDITS_PER_CELL_PER_YEAR = 20          # how often the average cell is rewritten

state_per_doc = CELLS_PER_DOC * CELL_STATE_BYTES
log_per_doc_per_year = CELLS_PER_DOC * EDITS_PER_CELL_PER_YEAR * OP_BYTES

table([
    ("Live state per doc",       human_bytes(state_per_doc)),
    ("Log per doc per year",     human_bytes(log_per_doc_per_year)),
    ("Compaction ratio",         f"{log_per_doc_per_year / state_per_doc:.0f} : 1"),
    ("", ""),
    ("Total live state",         human_bytes(state_per_doc * DOCS)),
    ("Total log, 1 year",        human_bytes(log_per_day * 365)),
], title="STATE vs LOG")

assert log_per_doc_per_year / state_per_doc > 20
print("\n  => The log is ~50x the state it produces. Snapshotting is not an")
print("     optimisation, it is what makes document LOAD possible at all:")
print("     fetch one snapshot, then only the ops after its version.")

# What load actually costs, with and without snapshots.
ops_since_snapshot = 200
table([
    ("Replay from scratch",  f"{CELLS_PER_DOC * EDITS_PER_CELL_PER_YEAR:,} ops"),
    ("Snapshot + tail",      f"1 fetch + {ops_since_snapshot} ops"),
    ("Speedup",              f"{CELLS_PER_DOC * EDITS_PER_CELL_PER_YEAR / ops_since_snapshot:.0f}x"),
], title="OPENING A ONE-YEAR-OLD DOCUMENT")
print("\n  Retention then has two different answers: the LOG can be truncated behind")
print("  the snapshot for correctness, but version history is a product feature -")
print("  so keep the log as long as you promise 'see version history', not longer.")

## Discussion — the follow-ups

- **Two users typing in the same cell.** Cell-level LWW means one of them loses their whole entry, not one character — which is jarring while you watch it happen. Google Sheets' answer is presence: show that someone else is editing the cell, and don't merge. If you genuinely want character-level merging you need OT or a text CRDT *inside* the cell, and then a cell is a small collaborative document of its own. Say which you're building; they are very different systems.
- **Two users formatting the same cell.** This is the argument for splitting `setValue` and `setFormat` into separate operation types. As one operation, bolding a cell and typing in it conflict for no reason — and worse, an operation carrying `format: null` silently clears formatting someone else just applied. Separate types make them independent, and independent operations never conflict.
- **Circular references.** The dependency graph must be checked for cycles *before* accepting the formula, not discovered during evaluation. That is [Kahn's algorithm](../../2.%20Coding_Questions/11.%20Task_Scheduler/11.%20Task_Scheduler.ipynb) again, or a DFS with a visiting set — and the failure mode is the same: validate at write time or hang at read time. Surface it as `#REF!` in the cell rather than rejecting the keystroke, because a user often builds a cycle *en route* to a valid formula.
- **Copy-paste of a range.** One operation, not N — otherwise a 10,000-cell paste becomes 10,000 broadcasts and every collaborator watches the paste dribble in. But a range operation overlapping concurrent single-cell edits needs a defined rule: simplest is to treat the range as one atomic entry in the total order, so it wins over everything before it and loses to everything after.
- **A 100,000-row sheet.** Don't send it. The client renders a viewport, so it should fetch a viewport — request cells by rectangle, with the sparse map making empty regions free. Note this interacts with the operation stream: a client must still receive operations for cells it isn't currently displaying, or it will silently miss updates when the user scrolls. Either subscribe to the whole document's op stream (cheap — ops are small) and materialise lazily, or re-fetch the region on scroll and accept a seam.

## Patterns learned

- **Store operations, derive state.** Durability, ordering, conflict resolution, and time travel all fall out of one decision. State-based sync gives you none of them.
- **One atomic counter can replace a consensus protocol.** A per-document monotonic `seq` gives a total order, and LWW on top of it is well-defined without any agreement between clients.
- **Ask what the ordering key actually measures.** `seq` records *arrival*, not *authorship*. Online those coincide; offline they don't, and every property built on `seq` inherits the gap.
- **Silent data loss is the worst bug class.** The offline path here loses reviewed work with no warning, and only for the same cell — which is exactly the case that survives casual testing.
- **A materialized view needs the key its consistency rule is defined on.** `cells` without `seq` cannot enforce "higher seq wins", so replay and out-of-order delivery corrupt it.
- **Idempotent and commutative are properties you build, not adjectives you assert.** `max(seq)`-wins makes them true; "commutative enough for our purposes" makes them nothing.
- **Count egress, not just ingress.** In any broadcast system, bandwidth is `ops × collaborators`. Storage compaction fixes the log; nothing fixes fan-out but sharding.
- **Snapshot + log tail is what makes load possible.** The log is ~50× the state it produces; replaying a year to open a document is not an option.
- **Split operation types that are independent.** Value and format edits conflicting with each other is an artifact of packing them into one operation.
- **Validate graphs at write time.** Cycle detection before accepting a formula, not during evaluation.